# 01 - Data Understanding

## Project

IT Service Performance Analytics

---

## Business Context

This project simulates a real-world Data Analyst mission within an IT Service Management (ITSM) department.

The objective of this notebook is to understand the structure, quality and business value of the ServiceNow incident management dataset before performing any cleaning, modeling or dashboard development.

---

## Objectives

- Load the raw incident event log dataset
- Explore the dataset structure and dimensions
- Assess data quality and missing values
- Detect duplicate records
- Understand variable types and business meaning
- Identify the columns required for future KPIs

---

## Expected Deliverables

- Dataset overview
- Data quality assessment
- Missing values analysis
- Data type analysis
- Initial business observations


## Table of Contents

1. [Import Libraries](#import-libraries)
2. [Load Dataset](#load-dataset)
3. [Dataset Dimensions](#dataset-dimensions)
4. [Dataset Preview](#dataset-preview)
5. [Dataset Information](#dataset-information)
6. [Data Types](#data-types)
7. [Missing Values Analysis](#missing-values-analysis)
8. [Duplicate Records](#duplicate-records)
9. [Descriptive Statistics](#descriptive-statistics)
10. [Variable Overview](#variable-overview)
11. [Key Variables for Business Analysis](#key-variables-for-business-analysis)
12. [Initial Business Observations](#initial-business-observations)
13. [Conclusion](#conclusion)


<a id="import-libraries"></a>

# Import Libraries

The analysis starts by importing the Python libraries required for data manipulation and exploration.

In [1]:
# ==========================================================
# Import libraries
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


<a id="load-dataset"></a>

# Load Dataset

The dataset is stored in the **data/raw** folder. The raw file uses `?` to represent missing values, so these placeholders are converted to proper null values when loading the file.

In [ ]:
data_path = Path("../data/raw/incident_event_log.csv")

if not data_path.exists():
    data_path = Path("data/raw/incident_event_log.csv")

df = pd.read_csv(
    data_path,
    na_values=["?"],
    dtype={'vendor': 'string'},
    low_memory=False,
)


C:\Users\djami\AppData\Local\Temp\ipykernel_25396\3624933777.py:6: DtypeWarning: Columns (0: cmdb_ci, 1: caused_by) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, na_values=["?"])


<a id="dataset-dimensions"></a>

# Dataset Dimensions

Understanding the size of the dataset helps evaluate whether the data is sufficient for operational performance analysis.

In [3]:
rows, columns = df.shape
unique_incidents = df["number"].nunique()

print(f"Rows: {rows:,}")
print(f"Columns: {columns}")
print(f"Unique incidents: {unique_incidents:,}")


Rows: 141,712
Columns: 36
Unique incidents: 24,918


## Business Interpretation

The dataset contains event-level records for IT incidents. This means that one incident can appear multiple times as it moves through different states or receives updates.

This level of detail is valuable for analyzing operational workflows, SLA compliance and incident lifecycle behavior.

<a id="dataset-preview"></a>

# Dataset Preview

A first preview helps identify the available fields and understand how the incident data is structured.

In [4]:
df.head()


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 56,NaN,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,NaN,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


## Business Interpretation

The dataset includes incident identifiers, workflow states, timestamps, SLA status, priority, assignment information and resolution details.

These fields can support future KPIs such as incident volume, SLA compliance rate, resolution time, reassignment frequency and support team performance.

<a id="dataset-information"></a>

# Dataset Information

The general dataset information provides a first view of column completeness and inferred data types.

In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 141712 entries, 0 to 141711
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   number                   141712 non-null  str  
 1   incident_state           141712 non-null  str  
 2   active                   141712 non-null  bool 
 3   reassignment_count       141712 non-null  int64
 4   reopen_count             141712 non-null  int64
 5   sys_mod_count            141712 non-null  int64
 6   made_sla                 141712 non-null  bool 
 7   caller_id                141683 non-null  str  
 8   opened_by                136877 non-null  str  
 9   opened_at                141712 non-null  str  
 10  sys_created_by           88636 non-null   str  
 11  sys_created_at           88636 non-null   str  
 12  sys_updated_by           141712 non-null  str  
 13  sys_updated_at           141712 non-null  str  
 14  contact_type             141712 non-null  str  

## Business Interpretation

This step helps identify:

- numerical variables such as reassignment and reopen counts;
- categorical variables such as priority, impact, urgency and assignment group;
- date variables currently stored as text;
- variables with missing values that will need cleaning rules.

<a id="data-types"></a>

# Data Types

Data types must be reviewed before analysis because incorrect types can lead to wrong calculations or misleading KPIs.

In [6]:
data_types = df.dtypes.reset_index()
data_types.columns = ["column", "data_type"]

data_types


,column,data_type
0,number,str
1,incident_state,str
2,active,bool
3,reassignment_count,int64
4,reopen_count,int64
5,sys_mod_count,int64
6,made_sla,bool
7,caller_id,str
8,opened_by,str
9,opened_at,str


## Business Interpretation

Several timestamp columns are currently stored as text. These columns will need to be converted to datetime format during the Data Cleaning phase before calculating resolution time, closure delays or monthly incident trends.

<a id="missing-values-analysis"></a>

# Missing Values Analysis

Missing values must be identified early because they can affect KPI reliability, segmentation and dashboard filters.

In [7]:
missing_values = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

missing_values.columns = ["column", "missing_values"]
missing_values["missing_percentage"] = (missing_values["missing_values"] / len(df) * 100).round(2)

missing_values


,column,missing_values,missing_percentage
0,caused_by,141689,99.98
1,vendor,141468,99.83
2,cmdb_ci,141267,99.69
3,rfc,140721,99.30
4,problem_id,139417,98.38
5,sys_created_by,53076,37.45
6,sys_created_at,53076,37.45
7,u_symptom,32964,23.26
8,assigned_to,27496,19.40
9,assignment_group,14213,10.03


In [8]:
missing_values.query("missing_values > 0")


,column,missing_values,missing_percentage
0,caused_by,141689,99.98
1,vendor,141468,99.83
2,cmdb_ci,141267,99.69
3,rfc,140721,99.30
4,problem_id,139417,98.38
5,sys_created_by,53076,37.45
6,sys_created_at,53076,37.45
7,u_symptom,32964,23.26
8,assigned_to,27496,19.40
9,assignment_group,14213,10.03


## Business Interpretation

Some fields are expected to be incomplete in an ITSM process. For example, vendor, change request or problem fields may only be populated when an incident is linked to external suppliers, changes or known problems.

High missingness does not automatically mean poor data quality. Each field must be evaluated according to its business role before deciding whether to keep, transform or exclude it.

<a id="duplicate-records"></a>

# Duplicate Records

Duplicate records are checked to identify whether the dataset contains repeated rows.

In [9]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")


Duplicate rows: 0


## Business Interpretation

At this stage, duplicate records are only identified. No records should be removed before confirming whether repeated information represents true duplicates or valid event history.

<a id="descriptive-statistics"></a>

# Descriptive Statistics

Descriptive statistics provide an initial view of numerical distributions and categorical dominance.

In [10]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
reassignment_count,141712.0,1.104197,1.734673,0.0,0.0,1.0,1.0,27.0
reopen_count,141712.0,0.021918,0.207302,0.0,0.0,0.0,0.0,8.0
sys_mod_count,141712.0,5.080946,7.680652,0.0,1.0,3.0,6.0,129.0


In [11]:
df.describe(include="object").T


C:\Users\djami\AppData\Local\Temp\ipykernel_25396\3164331651.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object").T


,count,unique,top,freq
number,141712,24918,INC0019396,58
incident_state,141712,9,Active,38716
caller_id,141683,5244,Caller 1904,1425
opened_by,136877,207,Opened by 17,41466
opened_at,141712,19849,14/4/2016 20:42,58
sys_created_by,88636,185,Created by 10,24223
sys_created_at,88636,11552,4/7/2016 14:50,56
sys_updated_by,141712,846,Updated by 908,36162
sys_updated_at,141712,50664,24/3/2016 18:40,420
contact_type,141712,5,Phone,140462


## Business Interpretation

Numerical statistics help detect potential outliers in workflow activity, such as incidents with many reassignments or reopenings.

Categorical statistics reveal dominant states, priorities, contact channels and support groups, which will be useful for operational dashboards.

<a id="variable-overview"></a>

# Variable Overview

A consolidated variable overview helps document the dataset and prepare decisions for cleaning and modeling.

In [12]:
variable_overview = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_values": df.isna().sum().values,
    "missing_percentage": (df.isna().sum().values / len(df) * 100).round(2),
    "unique_values": df.nunique(dropna=True).values
})

variable_overview


,column,data_type,missing_values,missing_percentage,unique_values
0,number,str,0,0.00,24918
1,incident_state,str,0,0.00,9
2,active,bool,0,0.00,2
3,reassignment_count,int64,0,0.00,28
4,reopen_count,int64,0,0.00,9
5,sys_mod_count,int64,0,0.00,115
6,made_sla,bool,0,0.00,2
7,caller_id,str,29,0.02,5244
8,opened_by,str,4835,3.41,207
9,opened_at,str,0,0.00,19849


## Business Interpretation

This table provides a data dictionary starting point. It highlights which variables are identifiers, dimensions, operational measures or lifecycle timestamps.

It will support the next phase by showing which fields require type conversion, missing value treatment or feature engineering.

<a id="key-variables-for-business-analysis"></a>

# Key Variables for Business Analysis

The following variables are expected to become central to SLA monitoring and IT service performance analysis.

In [13]:
business_columns = [
    "number",
    "incident_state",
    "priority",
    "impact",
    "urgency",
    "assignment_group",
    "reassignment_count",
    "reopen_count",
    "made_sla",
    "opened_at",
    "resolved_at",
    "closed_at"
]

df[business_columns].head()


,number,incident_state,priority,impact,urgency,assignment_group,reassignment_count,reopen_count,made_sla,opened_at,resolved_at,closed_at
0,INC0000045,New,3 - Moderate,2 - Medium,2 - Medium,Group 56,0,0,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,3 - Moderate,2 - Medium,2 - Medium,Group 56,0,0,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,3 - Moderate,2 - Medium,2 - Medium,Group 56,0,0,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,3 - Moderate,2 - Medium,2 - Medium,Group 56,0,0,True,29/2/2016 01:16,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,3 - Moderate,2 - Medium,2 - Medium,Group 70,0,0,True,29/2/2016 04:40,1/3/2016 09:52,6/3/2016 10:00


## Business Interpretation

These fields can support the main business indicators of the project:

- SLA compliance rate;
- incident volume by priority and category;
- average resolution time;
- reassignment and reopen rates;
- support group workload and performance.

<a id="initial-business-observations"></a>

# Initial Business Observations

This section summarizes the first business-oriented findings from the raw dataset.

In [14]:
initial_observations = pd.DataFrame({
    "metric": [
        "event_records",
        "unique_incidents",
        "columns",
        "duplicate_rows",
        "sla_compliant_events",
        "sla_breached_events"
    ],
    "value": [
        len(df),
        df["number"].nunique(),
        df.shape[1],
        duplicate_rows,
        df["made_sla"].eq(True).sum() if df["made_sla"].dtype == "bool" else df["made_sla"].astype(str).str.lower().eq("true").sum(),
        df["made_sla"].eq(False).sum() if df["made_sla"].dtype == "bool" else df["made_sla"].astype(str).str.lower().eq("false").sum()
    ]
})

initial_observations


,metric,value
0,event_records,141712
1,unique_incidents,24918
2,columns,36
3,duplicate_rows,0
4,sla_compliant_events,132497
5,sla_breached_events,9215


## Initial Findings

The dataset contains more than 140,000 event records related to nearly 25,000 IT incidents.

The data is rich enough to analyze IT service performance, but several preparation steps are required before building reliable KPIs.

Key preparation needs include converting date fields, handling missing values, validating event-level granularity and creating incident-level metrics where required.

<a id="conclusion"></a>

# Conclusion

The Data Understanding phase confirms that the raw ServiceNow incident event log contains the information required to support enterprise-level IT service analytics.

The next phase will focus on:

- converting date columns to datetime format;
- standardizing missing values;
- validating categorical fields;
- preparing incident-level analytical tables;
- engineering KPI-ready variables such as resolution time and SLA breach indicators.
